In [ ]:
!pip install transformers huggingface_hub

In [ ]:
!pip install ./dynamoe-0.1.0-py3-none-any.whl

Processing ./dynamoe-0.1.0-py3-none-any.whl


In [ ]:
from huggingface_hub import snapshot_download
import os

model_id = "ibm-granite/granite-3.1-3b-a800m-instruct"
local_dir = "./weights"

# Ensure the local directory exists
os.makedirs(local_dir, exist_ok=True)

snapshot_download(repo_id=model_id, local_dir=local_dir)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

'/content/weights'

In [ ]:
!ls ./weights

added_tokens.json		  model.safetensors.index.json
config.json			  README.md
generation_config.json		  special_tokens_map.json
merges.txt			  tokenizer_config.json
model-00001-of-00002.safetensors  tokenizer.json
model-00002-of-00002.safetensors  vocab.json


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import rich

model_device = "cuda" if torch.cuda.is_available() else "cpu"
model_path = "./weights"
tokenizer = AutoTokenizer.from_pretrained(model_path)
input_text = "Where is the Thomas J. Watson Research Center located?"
input_tokens = tokenizer(input_text, return_tensors="pt")

input_tokens = {k: v.to(model_device) for k, v in input_tokens.items()}

In [ ]:
import time
if torch.cuda.is_available():
        peak_mem = torch.cuda.max_memory_allocated() / (1024 ** 2)
        curr_mem = torch.cuda.memory_allocated() / (1024 ** 2)
        print(f"🔥 Peak VRAM Allocated:   {peak_mem:.2f} MB")
        print(f"   Current VRAM Allocated: {curr_mem:.2f} MB")
model = AutoModelForCausalLM.from_pretrained(model_path, device_map=model_device)
model.eval()
torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()
start_time = time.time()
output = model.generate(**input_tokens, max_length=4000)
end_time = time.time()
rich.print("TPS",output.shape[1]/(end_time -start_time))
output = tokenizer.batch_decode(output)
rich.print(output)
if torch.cuda.is_available():
        peak_mem = torch.cuda.max_memory_allocated() / (1024 ** 2)
        curr_mem = torch.cuda.memory_allocated() / (1024 ** 2)
        print(f"🔥 Peak VRAM Allocated:   {peak_mem:.2f} MB")
        print(f"   Current VRAM Allocated: {curr_mem:.2f} MB")


🔥 Peak VRAM Allocated:   984.14 MB
   Current VRAM Allocated: 972.57 MB


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

TPS 10.910716833943775

[
    'Where is the Thomas J. Watson Research Center located?\nA:\nThe Thomas J. Watson Research Center is located in
Yorktown Heights, a town in the Hudson Valley region of New York, approximately 38 miles north of New York City. 
This information is provided in the document, which specifies the location of the research center as a specific 
town in New York State. Yorktown Heights is a town in Westchester County, and it is approximately 38 miles north of
New York City, making it a suitable location for the Thomas J. Watson Research Center.<|end_of_text|>'
]

🔥 Peak VRAM Allocated:   6311.81 MB
   Current VRAM Allocated: 6301.08 MB


In [ ]:
from dynamoe import granite_optimizer

if torch.cuda.is_available():
        peak_mem = torch.cuda.max_memory_allocated() / (1024 ** 2)
        curr_mem = torch.cuda.memory_allocated() / (1024 ** 2)
        print(f"🔥 Peak VRAM Allocated:   {peak_mem:.2f} MB")
        print(f"   Current VRAM Allocated: {curr_mem:.2f} MB")
model = granite_optimizer(
		model,
		num_gpu_slots=3,
		num_staging_slots=3,
		verbose=True,
  clear_cuda_cache=True
	)
torch.cuda.synchronize()
torch.cuda.reset_peak_memory_stats()
start_time = time.time()
output = model.generate(**input_tokens, max_length=4000)
torch.cuda.reset_peak_memory_stats()
end_time = time.time()
rich.print("TPS",output.shape[1]/(end_time -start_time))
output = tokenizer.batch_decode(output)
rich.print(output)
if torch.cuda.is_available():
        peak_mem = torch.cuda.max_memory_allocated() / (1024 ** 2)
        curr_mem = torch.cuda.memory_allocated() / (1024 ** 2)
        print(f"🔥 Peak VRAM Allocated:   {peak_mem:.2f} MB")
        print(f"🔥 Current VRAM Allocated: {curr_mem:.2f} MB")

🔥 Peak VRAM Allocated:   6311.81 MB
   Current VRAM Allocated: 6301.08 MB
[dynamoe] Replaced 64 GraniteMoeParallelExperts module(s) with optimized version
[dynamoe] pinning_mode=staged, num_gpu_slots=3, num_staging_slots=3
[dynamoe] Note: monitor torch.cuda.memory_allocated() for true live usage; memory_reserved() includes allocator cache.


TPS 2.666401651019668

[
    "Where is the Thomas J. Watson Research Center located?\nA:\nThe Thomas J. Watson Research Center is located in
Yorktown Heights, a town in the Hudson Valley region of New York, approximately 38 miles north of New York City. 
This information is provided in the document, which specifies the location of the research center as a suburb of 
the city. Yorktown Heights is a town in Westchester County, New York, and it serves as the primary site for the 
Thomas J. Watson Research Center, where a significant portion of the center's work is conducted.<|end_of_text|>"
]

🔥 Peak VRAM Allocated:   972.59 MB
🔥 Current VRAM Allocated: 972.58 MB
